<a href="https://colab.research.google.com/github/666junyichen/Google-workshop/blob/main/Mini_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ImagineX Mini Challenge - Poster Creator
# Works in Google Colab and in a normal local Python notebook/script.

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

try:
    from IPython.display import display
except ImportError:
    display = None

try:
    from google.colab import files
except ImportError:
    files = None

# Step 1: Type your idea below (one sentence)
topic = input("Describe your idea in one sentence: ").strip() or "My workshop poster"

# Step 2: Choose mode
# Type A to upload/use your own image OR B to auto-generate a simple drawing
mode_prompt = "Choose mode: A = Upload/use image, B = Simple drawing  --> "
MODE = (input(mode_prompt).strip().upper() or "B")
if MODE not in {"A", "B"}:
    print("Unknown mode, using simple drawing instead.")
    MODE = "B"

# --- fonts ---
def F(name="DejaVuSans.ttf", size=42, bold=False):
    try:
        if bold:
            name = "DejaVuSans-Bold.ttf"
        return ImageFont.truetype(name, size)
    except OSError:
        return ImageFont.load_default()

# --- helpers ---
def fit_cover(img, size):
    target_w, target_h = size
    scale = max(target_w / img.width, target_h / img.height)
    new_size = (round(img.width * scale), round(img.height * scale))
    img = img.resize(new_size)
    left = (img.width - target_w) // 2
    top = (img.height - target_h) // 2
    return img.crop((left, top, left + target_w, top + target_h))

def draw_centered_text(draw, text, y, font, fill, width):
    bbox = draw.textbbox((0, 0), text, font=font)
    x = (width - (bbox[2] - bbox[0])) // 2
    draw.text((x, y), text, font=font, fill=fill)

# --- colours ---
CREAM = (246, 242, 232)
GREEN = (40, 120, 90)
SKY = (201, 226, 236)
LEAF1 = (141, 185, 149)
LEAF2 = (109, 160, 138)
LEAF3 = (85, 137, 118)
BROWN = (92, 62, 44)
YELLOW = (244, 214, 60)
SKIN = (232, 190, 150)
DARK = (34, 34, 34)

# --- make poster ---
W, H = 360, 562
poster = Image.new("RGB", (W, H), (28, 28, 28))
d = ImageDraw.Draw(poster)
d.rounded_rectangle([(8, 8), (W - 8, H - 8)], radius=18, fill=CREAM)

if MODE == "A":
    print("Upload a photo/artwork in Colab, or type a local image path here.")
    img = None
    if files is not None:
        uploaded = files.upload()
        if uploaded:
            fname = next(iter(uploaded.keys()))
            img = Image.open(fname).convert("RGB")
    else:
        image_path = input("Local image path (leave blank for simple drawing): ").strip().strip('"')
        if image_path:
            img = Image.open(image_path).convert("RGB")

    if img is None:
        print("No image selected, using simple drawing instead.")
        MODE = "B"
    else:
        win = (22, 22, W - 22, 410)
        wx1, wy1, wx2, wy2 = win
        img = fit_cover(img, (wx2 - wx1, wy2 - wy1))
        poster.paste(img, (wx1, wy1))

if MODE == "B":
    d.rectangle([(22, 22), (W - 22, 410)], fill=SKY)
    d.text((30, 36), "YOUR", font=F(size=48, bold=True), fill=GREEN)
    d.text((30, 96), "POSTER", font=F(size=48, bold=True), fill=GREEN)

    def tree(x, y, s=1.0):
        d.ellipse([(x - 46 * s, y - 26 * s), (x + 46 * s, y + 64 * s)], fill=LEAF1)
        d.ellipse([(x - 80 * s, y - 20 * s), (x - 10 * s, y + 50 * s)], fill=LEAF2)
        d.ellipse([(x + 12 * s, y - 12 * s), (x + 80 * s, y + 56 * s)], fill=LEAF3)
        d.rectangle([(x - 6 * s, y + 44 * s), (x + 6 * s, y + 116 * s)], fill=BROWN)

    tree(100, 250, 1.0)
    tree(260, 255, 1.0)
    tree(300, 280, 0.8)
    d.ellipse([(140, 210), (190, 255)], fill=SKIN)
    d.rectangle([(118, 268), (212, 320)], fill=YELLOW)
    d.polygon([(96, 336), (236, 336), (248, 392), (84, 392)], fill=DARK)

# bottom labels
draw_centered_text(d, topic[:28], 426, F(size=20, bold=True), DARK, W)
d.text((46, 466), "Date", font=F(size=24, bold=True), fill=GREEN)
d.text((158, 466), "Time", font=F(size=24, bold=True), fill=GREEN)
d.text((268, 466), "Place", font=F(size=24, bold=True), fill=GREEN)
for x in [46, 158, 268]:
    d.line([(x, 500), (x + 56, 500)], fill=GREEN, width=5)

# show + save
if display is not None:
    display(poster)

output_dir = Path("/content") if Path("/content").exists() else Path.cwd()
output_path = output_dir / "your_poster.png"
poster.save(output_path, quality=95)
print(f"\nPoster saved to {output_path}")
print(f"Your idea: {topic}")
